# CNN Architecture Diagram

This notebook generates the thesis-facing CNN architecture diagram from the active frozen and fine-tuned CNN setup. It does not train or evaluate the model. The constants mirror the active CNN notebooks.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Ellipse, FancyArrowPatch

PROJECT_ROOT = Path(os.environ.get("MDPP_ROOT", Path.cwd() / "mdpp_v2"))
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = ARTIFACTS_DIR / "cnn_architecture_diagram.png"
OUTPUT_PATH

In [ ]:
# Values copied from the active CNN notebooks.
CLASS_NAMES = [
    "Cotton",
    "Natural polyamide",
    "Polyester / recycled polyester / PET",
    "Regenerated cellulosic",
    "Synthetic polyamide / nylon",
]
FEATURE_DIM = 1920
OUTPUT_UNITS = len(CLASS_NAMES)
UNFROZEN_MODULES = ("features.denseblock4", "features.norm5")
TRAINABLE_PARAMETER_COUNTS = {
    "features.denseblock4": 6978560,
    "features.norm5": 3840,
    "classifier": 9605,
}

assert FEATURE_DIM == 1920
assert OUTPUT_UNITS == 5
assert UNFROZEN_MODULES == ("features.denseblock4", "features.norm5")
TRAINABLE_PARAMETER_COUNTS

In [ ]:
def rect(ax, x, y, w, h, text, fc, ec="#4f6478", lw=1.0, fontsize=8.9, rotation=0):
    patch = Rectangle((x, y), w, h, facecolor=fc, edgecolor=ec, linewidth=lw)
    ax.add_patch(patch)
    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        rotation=rotation,
        color="#1f2a36",
        linespacing=1.12,
    )
    return patch


def ellipse(ax, x, y, w, h, text, fc="#eef4fa", ec="#4f6478", fontsize=8.8):
    patch = Ellipse((x + w / 2, y + h / 2), w, h, facecolor=fc, edgecolor=ec, linewidth=1.0)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, color="#1f2a36", linespacing=1.08)
    return patch


def arrow(ax, start, end, color="#355f7c", width=1.25, scale=12):
    patch = FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=scale, linewidth=width, color=color)
    ax.add_patch(patch)
    return patch


fig, ax = plt.subplots(figsize=(13.2, 4.65))
ax.set_xlim(0, 13.2)
ax.set_ylim(-0.35, 4.5)
ax.axis("off")

# Palette kept intentionally small.
frozen_fill = "#f2f5f7"
trainable_fill = "#dcecf8"
output_fill = "#edf4fb"
frame_edge = "#7b8fa3"
arrow_color = "#355f7c"

# Outer groups.
rect(ax, 0.15, 0.55, 3.25, 3.45, "", "#ffffff", ec=frame_edge, lw=1.0)
rect(ax, 3.75, 0.55, 9.25, 3.45, "", "#ffffff", ec=frame_edge, lw=1.0)
ax.text(1.78, 4.16, "Input and preprocessing", ha="center", va="bottom", fontsize=11, fontweight="bold", color="#111827")
ax.text(8.38, 4.16, "DenseNet201 image model", ha="center", va="bottom", fontsize=11, fontweight="bold", color="#111827")

# Input formats.
ellipse(ax, 0.58, 3.10, 1.26, 0.48, "Colour\n(C)", fc=output_fill)
ellipse(ax, 0.58, 2.43, 1.26, 0.48, "Mono\n(M)", fc=output_fill)
ellipse(ax, 0.58, 1.76, 1.26, 0.48, "Polarized\n(P)", fc=output_fill)
ax.text(0.37, 2.46, "Microscopy\nformats", ha="center", va="center", rotation=90, fontsize=9.0, fontweight="bold", color="#111827")

# Preprocessing.
rect(ax, 2.22, 1.25, 0.36, 2.55, "L\nto\nRGB", frozen_fill, fontsize=8.3)
rect(ax, 2.80, 1.05, 0.50, 2.90, "DenseNet201\ntransform", frozen_fill, fontsize=8.3, rotation=90)

# Model path, matching DenseNet201Tail in notebook 06.
rect(ax, 4.05, 1.00, 1.42, 2.95, "Frozen\nDenseNet201\nprefix", frozen_fill, fontsize=9.3)
rect(ax, 5.90, 1.00, 0.88, 2.95, "denseblock4", trainable_fill, fontsize=9.1, rotation=90)
rect(ax, 7.02, 1.00, 0.36, 2.95, "norm5", trainable_fill, fontsize=8.5, rotation=90)
rect(ax, 7.62, 1.00, 0.42, 2.95, "ReLU", frozen_fill, fontsize=8.5, rotation=90)
rect(ax, 8.25, 1.00, 0.52, 2.95, "adaptive\navg pool\n+ flatten", frozen_fill, fontsize=8.2, rotation=90)
rect(ax, 9.02, 1.00, 0.98, 2.95, "Embedding\n1920", output_fill, fontsize=9.3)
rect(ax, 10.55, 1.00, 0.78, 2.95, "Linear head\n5 logits", trainable_fill, fontsize=8.8, rotation=90)
rect(ax, 11.92, 1.00, 0.64, 2.95, "Softmax", output_fill, fontsize=8.7, rotation=90)

# Output boxes with margin below model frame.
rect(ax, 8.66, -0.25, 1.70, 0.50, "Image retrieval vector", output_fill, fontsize=8.6)
rect(ax, 11.20, -0.25, 1.70, 0.50, "Class probabilities", output_fill, fontsize=8.6)

# Main arrows.
for y_center in (3.34, 2.67, 2.00):
    arrow(ax, (1.84, y_center), (2.22, y_center), color=arrow_color, width=1.15, scale=11)
arrow(ax, (2.58, 2.52), (2.80, 2.52), color=arrow_color, width=1.15, scale=11)
arrow(ax, (3.30, 2.52), (4.05, 2.52), color=arrow_color, width=1.45, scale=14)
arrow(ax, (5.47, 2.52), (5.90, 2.52), color=arrow_color, width=1.35, scale=13)
arrow(ax, (6.78, 2.52), (7.02, 2.52), color=arrow_color, width=1.15, scale=11)
arrow(ax, (7.38, 2.52), (7.62, 2.52), color=arrow_color, width=1.15, scale=11)
arrow(ax, (8.04, 2.52), (8.25, 2.52), color=arrow_color, width=1.15, scale=11)
arrow(ax, (8.77, 2.52), (9.02, 2.52), color=arrow_color, width=1.25, scale=12)
arrow(ax, (10.00, 2.52), (10.55, 2.52), color=arrow_color, width=1.35, scale=13)
arrow(ax, (11.33, 2.52), (11.92, 2.52), color=arrow_color, width=1.35, scale=13)

# Branch arrows.
arrow(ax, (9.51, 1.00), (9.51, 0.25), color=arrow_color, width=1.15, scale=11)
arrow(ax, (12.24, 1.00), (12.24, 0.25), color=arrow_color, width=1.15, scale=11)

# Notes and legend.
ax.text(4.76, 0.73, "frozen", ha="center", va="center", fontsize=8.2, style="italic", color="#374151")
ax.text(6.64, 0.73, "fine-tuned in iteration 2", ha="center", va="center", fontsize=8.2, style="italic", color="#374151")
ax.text(9.51, 0.72, "output shape: (1920)", ha="center", va="center", fontsize=8.1, style="italic", color="#374151")
ax.text(12.24, 0.72, "output shape: (5)", ha="center", va="center", fontsize=8.1, style="italic", color="#374151")


plt.tight_layout(pad=0.25)
fig.savefig(OUTPUT_PATH, dpi=260, bbox_inches="tight", facecolor="white")
plt.show()
print(OUTPUT_PATH)